In [3]:
# Janela Qt: widgets precisam de backend interativo.
%matplotlib qt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.animation import FuncAnimation
import pandas as pd

In [7]:
df = pd.read_csv("all_stocks_5yr.csv", parse_dates=["date"])
aapl = df[df["Name"] == "AAPL"].sort_values("date").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

(linha,) = ax.plot([], [], color="#1f6dc7", linewidth=1.8)
preenchimento = [None]  # mutável para usar dentro do closure

ax.set_xlim(aapl["date"].min(), aapl["date"].max())
ax.set_ylim(0, aapl["close"].max() * 1.08)
ax.set_title(
    "Animação — AAPL revelada dia a dia (2013–2018)",
    fontsize=14,
    pad=30,
    fontweight="bold",
)
ax.text(
    0.5,
    1.02,
    "FuncAnimation desenha a série progressivamente",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=10,
    color="#555555",
)
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.grid(axis="y", color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)

STEP = 20  # avança 20 dias por frame (=> ~60 frames)


def update(frame):
    fim = min(frame * STEP, len(aapl))
    x = aapl["date"][:fim]
    y = aapl["close"][:fim]
    linha.set_data(x, y)
    if preenchimento[0] is not None:
        preenchimento[0].remove()
    preenchimento[0] = ax.fill_between(x, y, 0, color="#cfe1f3", alpha=0.55)
    return linha, preenchimento[0]


anim = FuncAnimation(
    fig, update, frames=len(aapl) // STEP + 1, interval=40, blit=False, repeat=False
)

# Exportar (opcional):
anim.save('aapl_anim.gif', writer='pillow', fps=25)
#anim.save('aapl_anim.mp4', writer='ffmpeg', fps=30)

plt.tight_layout()
plt.show()